In [ ]:
import importlib
import sys

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

In [ ]:
# Load decision-labeled data
file_path_train = '../../../../../../data/Sepsis/tensor_data/decision_labeled/sepsis_all_5_train.pkl'
sepsis_train_dataset = torch.load(file_path_train, weights_only=False)

file_path_val = '../../../../../../data/Sepsis/tensor_data/decision_labeled/sepsis_all_5_val.pkl'
sepsis_val_dataset = torch.load(file_path_val, weights_only=False)

In [ ]:
# Sepsis dataset dynamic categories, features:
sepsis_all_categories = sepsis_train_dataset.all_categories

sepsis_all_categories_cat = sepsis_all_categories[0]
# print(sepsis_all_categories_cat)
sepsis_all_categories_num = sepsis_all_categories[1]
# print(sepsis_all_categories_num)
for i, cat in enumerate(sepsis_all_categories_cat):
     print(f"Sepsis dynamic categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(sepsis_all_categories_num):
     print(f"Sepsis dynamic numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of numerical: {num[1]}")
print('\n')
     
# Sepsis dataset static categories, features:
sepsis_all_stat_categories = sepsis_train_dataset.all_static_categories

sepsis_all_stat_categories_cat = sepsis_all_stat_categories[0]
# print(sepsis_all_stat_categories_cat)
sepsis_all_stat_categories_num = sepsis_all_stat_categories[1]
# print(sepsis_all_stat_categories_num)
for i, cat in enumerate(sepsis_all_stat_categories_cat):
     print(f"Sepsis static categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(sepsis_all_stat_categories_num):
     print(f"Sepsis static numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of numerical: {num[1]}")  

In [ ]:
# Create lists with name of Encoder features (input) and decoder features (input & output)
# Encoder features:
enc_feat_cat = []
enc_feat_num = []
for cat in sepsis_all_categories_cat:
    enc_feat_cat.append(cat[0])
for num in sepsis_all_categories_num:
    enc_feat_num.append(num[0])
enc_feat = [enc_feat_cat, enc_feat_num]
print("Input features encoder: ", enc_feat)

# Static encoder features:
stat_enc_feat_cat = []
stat_enc_feat_num = []
for cat in sepsis_all_stat_categories_cat:
     stat_enc_feat_cat.append(cat[0])
for num in sepsis_all_stat_categories_num:
     stat_enc_feat_num.append(num[0])
stat_enc_feat = [stat_enc_feat_cat, stat_enc_feat_num]
print("Input features static encoder: ", stat_enc_feat)

# Decoder features:
dec_feat_cat = ['concept:name']
dec_feat_num = []
dec_feat = [dec_feat_cat, dec_feat_num]
print("Features decoder: ", dec_feat)

# Guard tensors are pre-computed and stored in the dataset .pkl files
# (prepared during data loading via prepare_guard_tensors).
print(f"Guard tensors: targets {sepsis_train_dataset._guard_targets.shape}, "
      f"mask {sepsis_train_dataset._guard_mask.shape}, "
      f"confidence {sepsis_train_dataset._guard_confidence.shape}")

In [ ]:
import suffix_pred.models.K_UED_LSTM
importlib.reload(suffix_pred.models.K_UED_LSTM)
from suffix_pred.models.K_UED_LSTM import DropoutUncertaintyEncoderDecoderLSTM

# Prediction decoder output sequence length
seq_len_pred = sepsis_train_dataset.min_suffix_size

# Size hidden layer
hidden_size = 128

# Number of cells
num_layers = 4

# Fixed Dropout probability 
# Experiment with different values but be carefull with exploiting gradients:
dropout = 0.1

# Encoder Decoder model initialization
model = DropoutUncertaintyEncoderDecoderLSTM(data_set_categories=sepsis_all_categories,
                                             enc_feat=enc_feat,
                                             dec_feat=dec_feat,
                                             seq_len_pred=seq_len_pred,
                                             hidden_size=hidden_size,
                                             num_layers=num_layers,
                                             dropout=dropout,
                                             # static attributes
                                             static_data_set_categories=sepsis_all_stat_categories,
                                             static_enc_feat=stat_enc_feat
                                             )

In [ ]:
import suffix_pred.loss
importlib.reload(suffix_pred.loss)
from suffix_pred.loss import Loss

loss_obj = Loss()

In [ ]:
import suffix_pred.train
importlib.reload(suffix_pred.train)
from suffix_pred.train import UEDTrainer

from torch.optim.lr_scheduler import ReduceLROnPlateau

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Start learning rate
learning_rate = 1e-5
weight_decay = 0.0

# Optimizer and Scheduler
optimizer = torch.optim.AdamW(params=model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=15, min_lr=1e-8)


# Keep consistent across all models
# Epochs / Batch size
num_epochs = 100
batch_size = 128

# L2 regularization term
regularization_term = 1e-4

# Shuffle data
shuffle = True

# Scheduled sampling (same as clean baseline):
# TF ratio anneals from 1.0 → min_teacher_forcing_value via inverse-sigmoid.
# L_guard is applied only at decoder steps that consumed GT input (tf_mask=1).
teacher_forcing_mode = "scheduled"
max_teacher_forcing_value = 1.0
min_teacher_forcing_value = 0.0

optimize_values = {"regularization_term": regularization_term,
                   "optimizer": optimizer,
                   "scheduler": scheduler,
                   "epochs": num_epochs,
                   "mini_batches": batch_size,
                   "shuffle": shuffle,
                   "min_teacher_forcing_value": min_teacher_forcing_value,
                   "max_teacher_forcing_value": max_teacher_forcing_value,
                   "teacher_forcing_mode": teacher_forcing_mode}

required_optimize_keys = {"regularization_term",
                          "optimizer",
                          "scheduler",
                          "epochs",
                          "mini_batches",
                          "shuffle",
                          "min_teacher_forcing_value",
                          "max_teacher_forcing_value",
                          "teacher_forcing_mode"}

missing_keys = required_optimize_keys.difference(optimize_values.keys())
if missing_keys:
    raise ValueError(f"Missing required optimize_values keys: {sorted(missing_keys)}")

suffix_data_split_value = sepsis_train_dataset.min_suffix_size
print("Train seq length:", suffix_data_split_value)
print(f"Teacher forcing config: mode={teacher_forcing_mode}")

# Decision-aware guard regularization weight (λ_g)
lambda_g = 1.0
print(f"Decision-aware training: λ_g = {lambda_g}")

model_output_path = "../../../../../../models/Sepsis/decision/Sepsis_UED_LSTM_v2_DA.pkl"
os.makedirs(os.path.dirname(model_output_path), exist_ok=True)

trainer = UEDTrainer(device=device,
                     model=model,
                     data_train=sepsis_train_dataset,
                     data_val=sepsis_val_dataset,
                     loss_obj=loss_obj,
                     log_normal_loss_num_feature=[],
                     optimize_values=optimize_values,
                     suffix_data_split_value=suffix_data_split_value,
                     lambda_g=lambda_g,
                     save_model_n_th_epoch=1,
                     saving_path=model_output_path)

# Train the model (decision-aware: L_base + λ_g * L_guard)
train_attenuated_losses, val_losses, val_attenuated_losses = trainer.train_model(use_statics=True,
                                                                                 use_zero_padd_masking=True,
                                                                                 use_eos_padd_masking=True)